# Missing ModernMolBERT benchmark audit and optional rerun

This notebook checks whether each benchmark dataset is present for each ModernMolBERT model in `outputs/eval/best_metric_by_dataset_embedder.csv`. It also distinguishes three useful intermediate states: rows present in the raw per-model `results.csv`, rows present only in per-dataset checkpoints, and missing embedding files.

Manuscript note: the current manuscript is `paper/main.tex`. Its title and placeholder abstract frame the primary manuscript contribution as a ModernBERT encoder family for SELFIES molecular language modeling. For the internal ablation axis, `SPAN_MASKING_PLAN_V2.md` defines the ladder as `masking_strategy`: `standard` vs `span` vs `hetero_span`. The `base` model is a capacity/scale comparison, not the masking-strategy ablation axis.

In [1]:
from pathlib import Path
import shlex
import subprocess

import pandas as pd
from IPython.display import display


def find_repo_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for path in [start, *start.parents]:
        if (path / "pyproject.toml").exists() and (path / "src/modernmolbert").exists():
            return path
    raise RuntimeError("Could not locate ModernMolBERT repo root")


ROOT = find_repo_root()
BEST_CSV = ROOT / "outputs/eval/best_metric_by_dataset_embedder.csv"
EVAL_ROOT = ROOT / "outputs/eval"
DATASET_CONFIG = ROOT / "src/modernmolbert/eval/benchmarking_molecular_models/config/datasets.yaml"
SCORE_PY = ROOT / "src/modernmolbert/eval/benchmarking_molecular_models/score.py"
EMBED_PY = ROOT / "src/modernmolbert/eval/benchmarking_molecular_models/embed_modernmolbert.py"
BUILD_BENCHMARK_FRAMES_PY = ROOT / "scripts/build_benchmark_results_frames.py"

MODELS = [
    "modernmolbert_best_standard",
    "modernmolbert_best_base",
    "modernmolbert_best_span",
    "modernmolbert_best_hetero_span",
]

MODEL_TO_RESULT_DIR = {
    "modernmolbert_best_standard": EVAL_ROOT / "praski_best_standard",
    "modernmolbert_best_base": EVAL_ROOT / "praski_best_base_standard",
    "modernmolbert_best_span": EVAL_ROOT / "praski_best_span",
    "modernmolbert_best_hetero_span": EVAL_ROOT / "praski_best_hetero_span",
}

# Used only when RUN_MISSING_EMBEDDINGS is enabled below.
# NOTE: _standard and _span have weights under final_model/; _hetero_span has
# weights directly in the run dir; _base also has weights under final_model/.
MODEL_TO_MODEL_DIR = {
    "modernmolbert_best_standard": ROOT
    / "runs/chembl36_small_mask_mlm_lr_sweep/modernmolbert_best_standard/final_model",
    "modernmolbert_best_base": ROOT
    / "runs/chembl36_small_mask_mlm_lr_sweep/modernmolbert_best_base/final_model",
    "modernmolbert_best_span": ROOT
    / "runs/chembl36_small_mask_mlm_lr_sweep/modernmolbert_best_span/final_model",
    "modernmolbert_best_hetero_span": ROOT
    / "runs/chembl36_small_mask_mlm_lr_sweep/modernmolbert_best_hetero_span",
}

print(ROOT)

/Users/skn506/Documents/ModernMolBERT


In [2]:
HEADS = ["rf", "ridge", "knn"]
EXCLUDED_DATASETS = {"ogbg-moltoxcast"}


def load_best() -> pd.DataFrame:
    if not BEST_CSV.exists():
        raise FileNotFoundError(BEST_CSV)
    return pd.read_csv(BEST_CSV)


def safe_component(value: str) -> str:
    return str(value).replace("/", "_").replace("\\", "_").replace(":", "_").replace(" ", "_")


def result_csv(model: str) -> Path:
    return MODEL_TO_RESULT_DIR[model] / "results.csv"


def checkpoint_path(dataset: str, model: str) -> Path:
    return (
        MODEL_TO_RESULT_DIR[model]
        / "checkpoints"
        / f"{safe_component(dataset)}__{safe_component(model)}.csv"
    )


def head_json_dir(dataset: str, model: str) -> Path:
    """Per-head JSON checkpoint directory written by score.py (dataset/model/*.json)."""
    return (
        MODEL_TO_RESULT_DIR[model] / "checkpoints" / safe_component(dataset) / safe_component(model)
    )


def embedding_path(dataset: str, model: str) -> Path:
    return ROOT / "data/embedded" / dataset / f"{model}.joblib"


def dataset_selector(dataset: str) -> str:
    selector = f"clf_{dataset}"
    text = DATASET_CONFIG.read_text()
    if f"  {selector}:" in text:
        return selector
    if f"  {dataset}:" in text:
        return dataset
    raise KeyError(f"No dataset selector found for {dataset!r} in {DATASET_CONFIG}")


def csv_has_pair(path: Path, dataset: str, model: str) -> bool:
    if not path.exists() or path.stat().st_size == 0:
        return False
    try:
        df = pd.read_csv(path, usecols=["dataset", "embedder"])
    except ValueError:
        df = pd.read_csv(path)
    if not {"dataset", "embedder"}.issubset(df.columns):
        return False
    return bool(((df["dataset"] == dataset) & (df["embedder"] == model)).any())


def heads_in_csv(path: Path, dataset: str, model: str) -> frozenset[str]:
    """Return heads already present in a results or flat-checkpoint CSV."""
    if not path.exists() or path.stat().st_size == 0:
        return frozenset()
    try:
        df = pd.read_csv(path, usecols=["dataset", "embedder", "model"])
    except ValueError:
        return frozenset()
    if not {"dataset", "embedder", "model"}.issubset(df.columns):
        return frozenset()
    mask = (df["dataset"] == dataset) & (df["embedder"] == model)
    return frozenset(df.loc[mask, "model"].dropna().unique())


def heads_in_json_dir(dataset: str, model: str) -> frozenset[str]:
    """Return heads with a per-head JSON checkpoint (success *or* disabled).

    score.py writes per-head JSON files under
    checkpoints/<dataset>/<model>/<head>.json with status='success'|'disabled'.
    A disabled head (e.g. knn on MUV) will never produce a row in results.csv,
    so it should count as handled rather than missing.
    """
    import json as _json

    d = head_json_dir(dataset, model)
    if not d.exists():
        return frozenset()
    handled = set()
    for jf in d.glob("*.json"):
        try:
            payload = _json.loads(jf.read_text())
            status = payload.get("status", "")
            if status in ("success", "disabled"):
                handled.add(jf.stem)  # stem = head name (rf / ridge / knn)
        except Exception:
            pass
    return frozenset(handled)


def benchmark_present(dataset: str, model: str, *, source: str = "aggregate") -> bool:
    if source == "aggregate":
        df = load_best()
        return bool(((df["dataset"] == dataset) & (df["embedder"] == model)).any())
    if source == "raw":
        return csv_has_pair(result_csv(model), dataset, model)
    if source == "checkpoint":
        return csv_has_pair(checkpoint_path(dataset, model), dataset, model)
    raise ValueError(f"Unknown source: {source}")


def backfill_checkpoints_to_results() -> list[dict]:
    """Append flat-checkpoint CSV rows into results.csv for any
    result_not_in_aggregate cases.

    This is needed when the scorer wrote the flat per-dataset checkpoint CSV but
    the append to results.csv failed or was interrupted.  Safe to call repeatedly:
    rows are deduped by the 'key' column before appending.

    Returns a list of dicts describing what was backfilled.
    """
    coverage = audit_coverage()
    not_in_agg = coverage[coverage["status"] == "result_not_in_aggregate"]
    backfilled = []
    for _, row in not_in_agg.iterrows():
        ckpt = Path(ROOT) / row["checkpoint_csv"]
        res = Path(ROOT) / row["result_csv"]
        if not ckpt.exists():
            continue
        ckpt_df = pd.read_csv(ckpt)
        if len(ckpt_df) == 0:
            continue  # hollow checkpoint — needs re-scoring
        res_df = pd.read_csv(res) if res.exists() else pd.DataFrame()
        existing_keys = set(res_df["key"].dropna()) if "key" in res_df.columns else set()
        new_rows = ckpt_df[~ckpt_df["key"].isin(existing_keys)]
        if len(new_rows) == 0:
            continue
        combined = pd.concat([res_df, new_rows], ignore_index=True, sort=False)
        combined.to_csv(res, index=False)
        backfilled.append(
            {
                "dataset": row["dataset"],
                "model": row["model"],
                "rows_added": len(new_rows),
                "result_csv": str(row["result_csv"]),
            }
        )
        print(
            f"  backfilled {len(new_rows)} rows for {row['model']} / {row['dataset']} → {row['result_csv']}"
        )
    return backfilled


def audit_coverage() -> pd.DataFrame:
    best = load_best()
    datasets = sorted(d for d in best["dataset"].dropna().unique() if d not in EXCLUDED_DATASETS)
    records = []
    for model in MODELS:
        raw_path = result_csv(model)
        for dataset in datasets:
            in_aggregate = benchmark_present(dataset, model, source="aggregate")
            in_raw = benchmark_present(dataset, model, source="raw")
            in_checkpoint = benchmark_present(dataset, model, source="checkpoint")
            has_embedding = embedding_path(dataset, model).exists()

            # Per-head coverage: flat CSV + per-head JSON (success or disabled)
            raw_heads = heads_in_csv(raw_path, dataset, model)
            ckpt_heads = heads_in_csv(checkpoint_path(dataset, model), dataset, model)
            json_heads = heads_in_json_dir(dataset, model)
            present_heads = raw_heads | ckpt_heads | json_heads
            # Heads that still need scoring (not present in any source)
            missing_heads = frozenset(HEADS) - present_heads

            if in_aggregate:
                status = "present"
            elif in_raw or in_checkpoint:
                status = "result_not_in_aggregate"
            elif has_embedding:
                status = "missing_score_can_rerun"
            else:
                status = "missing_embedding"
            records.append(
                {
                    "dataset": dataset,
                    "model": model,
                    "in_aggregate": in_aggregate,
                    "in_raw_results": in_raw,
                    "in_checkpoint": in_checkpoint,
                    "has_embedding": has_embedding,
                    "status": status,
                    "heads_present": sorted(present_heads) or None,
                    "missing_heads": sorted(missing_heads) if missing_heads else None,
                    "dataset_selector": dataset_selector(dataset),
                    "result_csv": result_csv(model).relative_to(ROOT),
                    "checkpoint_csv": checkpoint_path(dataset, model).relative_to(ROOT),
                    "embedding_file": embedding_path(dataset, model).relative_to(ROOT),
                }
            )
    return pd.DataFrame.from_records(records)


coverage = audit_coverage()
display(coverage.groupby(["model", "status"]).size().unstack(fill_value=0))
display(coverage[coverage["status"] != "present"].sort_values(["model", "dataset"]))

status,missing_score_can_rerun,present
model,,
modernmolbert_best_base,1,24
modernmolbert_best_hetero_span,0,25
modernmolbert_best_span,0,25
modernmolbert_best_standard,1,24


,dataset,model,in_aggregate,in_raw_results,in_checkpoint,has_embedding,status,heads_present,missing_heads,dataset_selector,result_csv,checkpoint_csv,embedding_file
47,ogbg-molmuv,modernmolbert_best_base,False,False,False,True,missing_score_can_rerun,None,"[knn, rf, ridge]",clf_ogbg-molmuv,outputs/eval/praski_best_base_standard/results...,outputs/eval/praski_best_base_standard/checkpo...,data/embedded/ogbg-molmuv/modernmolbert_best_b...
22,ogbg-molmuv,modernmolbert_best_standard,False,False,False,True,missing_score_can_rerun,[knn],"[rf, ridge]",clf_ogbg-molmuv,outputs/eval/praski_best_standard/results.csv,outputs/eval/praski_best_standard/checkpoints/...,data/embedded/ogbg-molmuv/modernmolbert_best_s...


In [3]:
coverage_matrix = coverage.pivot(index="dataset", columns="model", values="status")
display(coverage_matrix[MODELS])

missing_by_model = (
    coverage.loc[~coverage["in_aggregate"]].groupby("model")["dataset"].apply(list).reindex(MODELS)
)
missing_by_model = missing_by_model.apply(
    lambda datasets: datasets if isinstance(datasets, list) else []
)

n_datasets = coverage["dataset"].nunique()
for model, datasets in missing_by_model.items():
    datasets_list = datasets if isinstance(datasets, list) else []
    print(
        f"{model}: {n_datasets - len(datasets_list)}/{n_datasets} present; missing={datasets_list}"
    )

model,modernmolbert_best_standard,modernmolbert_best_base,modernmolbert_best_span,modernmolbert_best_hetero_span
dataset,,,,
AMES,present,present,present,present
Bioavailability_Ma,present,present,present,present
CYP1A2_Veith,present,present,present,present
CYP2C19_Veith,present,present,present,present
CYP2C9_Substrate_CarbonMangels,present,present,present,present
CYP2C9_Veith,present,present,present,present
CYP2D6_Substrate_CarbonMangels,present,present,present,present
CYP2D6_Veith,present,present,present,present
CYP3A4_Substrate_CarbonMangels,present,present,present,present


modernmolbert_best_standard: 24/25 present; missing=['ogbg-molmuv']
modernmolbert_best_base: 24/25 present; missing=['ogbg-molmuv']
modernmolbert_best_span: 25/25 present; missing=[]
modernmolbert_best_hetero_span: 25/25 present; missing=[]


In [4]:
def list_missing(coverage: pd.DataFrame) -> None:
    """Print every missing (dataset, model, head) triple, grouped by model.

    Also prints copy-paste-ready RERUN_TARGETS entries for use in the cell below.
    """
    missing = coverage.loc[~coverage["in_aggregate"]].copy()

    if missing.empty:
        print("No missing evals — all datasets present in aggregate CSV.")
        return

    print(f"{'=' * 70}")
    print(f"MISSING EVALS  ({len(missing)} dataset/model pairs across all heads)")
    print(f"{'=' * 70}")

    rerun_lines: list[str] = []

    for model, group in missing.groupby("model", sort=False):
        print(f"\n  {model}  ({len(group)} datasets)")
        for row in group.sort_values("dataset").itertuples(index=False):
            missing_h = row.missing_heads if row.missing_heads is not None else list(HEADS)
            present_h = row.heads_present if row.heads_present is not None else []
            status_tag = f"[{row.status}]"
            print(f"    {row.dataset:<35} present={present_h}  missing={missing_h}  {status_tag}")

            if len(missing_h) == len(HEADS):
                # All heads missing — emit a 2-tuple
                rerun_lines.append(f'    ("{row.dataset}", "{model}"),')
            else:
                # Only specific heads missing — emit one 3-tuple per head
                for h in missing_h:
                    rerun_lines.append(f'    ("{row.dataset}", "{model}", "{h}"),')

    print(f"\n{'=' * 70}")
    print("RERUN_TARGETS snippet (copy into the cell below):")
    print("RERUN_TARGETS = [")
    for line in rerun_lines:
        print(line)
    print("]")


list_missing(coverage)

MISSING EVALS  (2 dataset/model pairs across all heads)

  modernmolbert_best_standard  (1 datasets)
    ogbg-molmuv                         present=['knn']  missing=['rf', 'ridge']  [missing_score_can_rerun]

  modernmolbert_best_base  (1 datasets)
    ogbg-molmuv                         present=[]  missing=['knn', 'rf', 'ridge']  [missing_score_can_rerun]

RERUN_TARGETS snippet (copy into the cell below):
RERUN_TARGETS = [
    ("ogbg-molmuv", "modernmolbert_best_standard", "rf"),
    ("ogbg-molmuv", "modernmolbert_best_standard", "ridge"),
    ("ogbg-molmuv", "modernmolbert_best_base"),
]


## Optional rerun controls

Default behavior is audit-only. Flip the relevant booleans to run subprocesses. Use `RERUN_TARGETS = "all_missing"` to loop over every missing benchmark row: rows without embeddings are embedded first, coverage is refreshed, and then all rows with embeddings can be scored. Scoring uses `--resume`, so dataset/model checkpoints that already exist are skipped instead of re-scored. After any scoring run, rebuild the aggregate CSV with `scripts/build_benchmark_results_frames.py`.

Set `KEEP_GOING_ON_ERROR = True` to continue through the remaining embeddings/scores if one subprocess fails. Failures are collected and printed at the end of the execution cell.


In [5]:
# Execution toggles. Keep these False for audit-only runs.
RUN_MISSING_EMBEDDINGS = True
RUN_MISSING_SCORE = True
REBUILD_AGGREGATES_AFTER_SCORE = True

# Backfill checkpoint rows that landed in per-dataset CSV but not in results.csv.
# This handles the case where the scorer was interrupted after writing the checkpoint
# but before appending to results.csv. Safe to run repeatedly (deduped by key).
RUN_BACKFILL = True

# If False, the first failed subprocess raises immediately.
# If True, failed subprocesses are recorded and the notebook continues with remaining targets.
KEEP_GOING_ON_ERROR = False

# Choose targets. Options:
#   "all_missing"                            — every non-aggregate row
#   [(dataset, model), ...]                  — specific dataset/model pairs; all heads scored
#   [(dataset, model, head), ...]            — specific dataset/model/head triples (passes --heads to scorer)
#
# Examples:
#   RERUN_TARGETS = [("ogbg-molhiv", "modernmolbert_best_span")]
#   RERUN_TARGETS = [("AMES", "modernmolbert_best_base", "knn")]
RERUN_TARGETS: str | list[tuple] = "all_missing"

# Embedding settings used only when RUN_MISSING_EMBEDDINGS is True.
EMBED_BATCH_SIZE = 32
EMBED_DEVICE = "auto"
EMBED_MAX_SEQ_LENGTH = 128
EMBED_POOLING = "mean"
OVERWRITE_EXISTING_EMBEDDINGS = False

# Cap RF/KNN parallelism to limit peak RAM. None = all cores (default).
# Set to e.g. 4 if the scorer swaps on large datasets (MUV, HIV).
N_JOBS_SCORER: int | None = 4


def selected_targets(coverage: pd.DataFrame) -> pd.DataFrame:
    missing = coverage.loc[~coverage["in_aggregate"]].copy()
    missing["target_heads"] = missing["missing_heads"].apply(
        lambda heads: list(heads) if heads is not None else []
    )

    if RERUN_TARGETS == "all_missing":
        return missing

    if isinstance(RERUN_TARGETS, str):
        raise ValueError('RERUN_TARGETS must be "all_missing" or a list of tuples')

    rows = []
    for entry in RERUN_TARGETS:
        if len(entry) == 2:
            dataset, model = entry
            matching = missing.loc[(missing["dataset"] == dataset) & (missing["model"] == model)]
            if matching.empty:
                head_filter = []
            else:
                heads = matching.iloc[0]["missing_heads"]
                head_filter = list(heads) if heads is not None else []
        elif len(entry) == 3:
            dataset, model, head = entry
            head_filter = [head]
        else:
            raise ValueError(f"RERUN_TARGETS entries must be 2- or 3-tuples, got: {entry!r}")
        mask = (missing["dataset"] == dataset) & (missing["model"] == model)
        subset = missing.loc[mask].copy()
        subset["target_heads"] = [head_filter] * len(subset)
        rows.append(subset)

    return pd.concat(rows, ignore_index=True) if rows else missing.iloc[:0].copy()


def split_targets_for_execution(targets: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Return (needs_embedding, ready_to_score) for the current coverage state."""
    targets_to_embed = targets.loc[~targets["has_embedding"]].copy()
    has_target_heads = targets["target_heads"].apply(bool)
    targets_to_score = targets.loc[targets["has_embedding"] & has_target_heads].copy()
    return targets_to_embed, targets_to_score


targets = selected_targets(coverage)
targets_to_embed, targets_to_score = split_targets_for_execution(targets)

print(f"selected missing targets: {len(targets)}")
print(f"targets needing embeddings first: {len(targets_to_embed)}")
print(f"targets with missing score heads ready to score now: {len(targets_to_score)}")
print(
    "If RUN_MISSING_EMBEDDINGS and RUN_MISSING_SCORE are both True, scoring is refreshed after embedding."
)
display(
    targets[
        ["dataset", "model", "status", "heads_present", "missing_heads", "target_heads"]
    ].sort_values(["model", "dataset"])
)

selected missing targets: 2
targets needing embeddings first: 0
targets with missing score heads ready to score now: 2
If RUN_MISSING_EMBEDDINGS and RUN_MISSING_SCORE are both True, scoring is refreshed after embedding.


,dataset,model,status,heads_present,missing_heads,target_heads
47,ogbg-molmuv,modernmolbert_best_base,missing_score_can_rerun,None,"[knn, rf, ridge]","[knn, rf, ridge]"
22,ogbg-molmuv,modernmolbert_best_standard,missing_score_can_rerun,[knn],"[rf, ridge]","[rf, ridge]"


In [6]:
def rel(path: Path) -> str:
    return str(path.relative_to(ROOT))


def score_command(dataset: str, model: str, heads: list[str] | None = None) -> list[str]:
    out_dir = MODEL_TO_RESULT_DIR[model]
    cmd = [
        "uv",
        "run",
        "python",
        rel(SCORE_PY),
        "--datasets",
        dataset_selector(dataset),
        "--embedder",
        model,
        "--output-csv",
        rel(out_dir / "results.csv"),
        "--checkpoint-dir",
        rel(out_dir / "checkpoints"),
        "--resume",
        "--safe",
    ]
    if N_JOBS_SCORER is not None:
        cmd += ["--n-jobs", str(N_JOBS_SCORER)]
    if heads and set(heads) != set(HEADS):
        cmd += ["--heads"] + heads
    return cmd


def embed_command(dataset: str, model: str) -> list[str]:
    model_dir = MODEL_TO_MODEL_DIR[model]
    cmd = [
        "uv",
        "run",
        "python",
        rel(EMBED_PY),
        "--datasets",
        dataset_selector(dataset),
        "--model-dir",
        rel(model_dir),
        "--tokenizer-path",
        rel(model_dir),
        "--embedder",
        model,
        "--batch-size",
        str(EMBED_BATCH_SIZE),
        "--device",
        EMBED_DEVICE,
        "--max-seq-length",
        str(EMBED_MAX_SEQ_LENGTH),
        "--pooling",
        EMBED_POOLING,
    ]
    if OVERWRITE_EXISTING_EMBEDDINGS:
        cmd.append("--overwrite")
    return cmd


def command_label(kind: str, dataset: str, model: str, heads: list[str] | None = None) -> str:
    if heads and set(heads) != set(HEADS):
        return f"{kind}: {model} / {dataset} / heads={heads}"
    return f"{kind}: {model} / {dataset}"


def run_command(
    cmd: list[str],
    *,
    enabled: bool,
    label: str | None = None,
    failures: list[dict[str, object]] | None = None,
) -> bool:
    printable = " ".join(shlex.quote(part) for part in cmd)
    if label:
        print(f"\n### {label}")
    print("$", printable)
    if not enabled:
        return True

    try:
        subprocess.run(cmd, cwd=ROOT, check=True)
        return True
    except subprocess.CalledProcessError as exc:
        if failures is not None:
            failures.append(
                {
                    "label": label or printable,
                    "returncode": exc.returncode,
                    "command": printable,
                }
            )
        if KEEP_GOING_ON_ERROR:
            print(
                f"FAILED with return code {exc.returncode}; continuing because KEEP_GOING_ON_ERROR=True"
            )
            return False
        raise


def print_failures(failures: list[dict[str, object]]) -> None:
    if not failures:
        print("No subprocess failures recorded.")
        return

    print(f"\n{len(failures)} subprocess failure(s):")
    for failure in failures:
        print(f"- {failure['label']}  returncode={failure['returncode']}")
        print(f"  {failure['command']}")


print("Embedding commands for missing embeddings:")
for row in targets_to_embed.sort_values(["model", "dataset"]).itertuples(index=False):
    run_command(
        embed_command(row.dataset, row.model),
        enabled=False,
        label=command_label("embed", row.dataset, row.model),
    )

print("\nScoring commands for targets with missing score heads now:")
for row in targets_to_score.sort_values(["model", "dataset"]).itertuples(index=False):
    run_command(
        score_command(row.dataset, row.model, row.target_heads),
        enabled=False,
        label=command_label("score", row.dataset, row.model, row.target_heads),
    )

if len(targets_to_embed) and RUN_MISSING_EMBEDDINGS and RUN_MISSING_SCORE:
    print(
        "\nAfter embedding, the execution cell refreshes coverage and scores newly embedded targets too."
    )

Embedding commands for missing embeddings:

Scoring commands for targets with missing score heads now:

### score: modernmolbert_best_base / ogbg-molmuv
$ uv run python src/modernmolbert/eval/benchmarking_molecular_models/score.py --datasets clf_ogbg-molmuv --embedder modernmolbert_best_base --output-csv outputs/eval/praski_best_base_standard/results.csv --checkpoint-dir outputs/eval/praski_best_base_standard/checkpoints --resume --safe --n-jobs 4

### score: modernmolbert_best_standard / ogbg-molmuv / heads=['rf', 'ridge']
$ uv run python src/modernmolbert/eval/benchmarking_molecular_models/score.py --datasets clf_ogbg-molmuv --embedder modernmolbert_best_standard --output-csv outputs/eval/praski_best_standard/results.csv --checkpoint-dir outputs/eval/praski_best_standard/checkpoints --resume --safe --n-jobs 4 --heads rf ridge


In [7]:
failures: list[dict[str, object]] = []

if RUN_MISSING_EMBEDDINGS:
    print(f"Embedding {len(targets_to_embed)} missing dataset/model pair(s).")
    for row in targets_to_embed.sort_values(["model", "dataset"]).itertuples(index=False):
        model_dir = MODEL_TO_MODEL_DIR[row.model]
        if not model_dir.exists():
            msg = f"Model directory does not exist: {model_dir}"
            if KEEP_GOING_ON_ERROR:
                print(f"FAILED: {msg}; continuing because KEEP_GOING_ON_ERROR=True")
                failures.append(
                    {
                        "label": command_label("embed", row.dataset, row.model),
                        "returncode": "missing_model_dir",
                        "command": msg,
                    }
                )
                continue
            raise FileNotFoundError(msg)

        run_command(
            embed_command(row.dataset, row.model),
            enabled=True,
            label=command_label("embed", row.dataset, row.model),
            failures=failures,
        )

    # Refresh so newly created embeddings become eligible for scoring.
    coverage = audit_coverage()
    targets = selected_targets(coverage)
    targets_to_embed, targets_to_score = split_targets_for_execution(targets)
    print(
        f"After embedding refresh: {len(targets_to_embed)} still need embeddings; {len(targets_to_score)} ready to score."
    )

if RUN_MISSING_SCORE:
    print(f"Scoring {len(targets_to_score)} dataset/model pair(s) with missing heads.")
    for row in targets_to_score.sort_values(["model", "dataset"]).itertuples(index=False):
        run_command(
            score_command(row.dataset, row.model, row.target_heads),
            enabled=True,
            label=command_label("score", row.dataset, row.model, row.target_heads),
            failures=failures,
        )

if RUN_BACKFILL:
    print("Backfilling checkpoint rows into results.csv for result_not_in_aggregate cases...")
    bf = backfill_checkpoints_to_results()
    if not bf:
        print("  Nothing to backfill.")

if REBUILD_AGGREGATES_AFTER_SCORE:
    run_command(
        ["uv", "run", "python", rel(BUILD_BENCHMARK_FRAMES_PY)],
        enabled=True,
        label="rebuild aggregate benchmark frames",
        failures=failures,
    )

print_failures(failures)

Embedding 0 missing dataset/model pair(s).
After embedding refresh: 0 still need embeddings; 2 ready to score.
Scoring 2 dataset/model pair(s) with missing heads.

### score: modernmolbert_best_base / ogbg-molmuv
$ uv run python src/modernmolbert/eval/benchmarking_molecular_models/score.py --datasets clf_ogbg-molmuv --embedder modernmolbert_best_base --output-csv outputs/eval/praski_best_base_standard/results.csv --checkpoint-dir outputs/eval/praski_best_base_standard/checkpoints --resume --safe --n-jobs 4
[score] embedder=modernmolbert_best_base  expanded_datasets=1  datasets_to_run=1  heads=['rf', 'ridge', 'knn']  override=False  safe=True  resume=True  subsample=disabled
[score] run plan:
  [ 1/1] ogbg-molmuv
[ 1/1] ogbg-molmuv  head=rf


2026-06-03 10:15:40,276 [INFO] Shape (93087, 768) for dataset ogbg-molmuv, task classification
2026-06-03 10:15:40,276 [INFO] Evaluating model modernmolbert_best_base on dataset ogbg-molmuv with metric roc_auc using head rf
2026-06-03 10:15:40,281 [INFO] Training model


Shapes: X_test=(9309, 768), y_test=(9309, 17)


2026-06-03 11:16:58,644 [INFO] Training complete, best CV result: 0.6903619119707421
2026-06-03 11:16:58,658 [INFO] Evaluation complete, test result: 0.7139028352463384
2026-06-03 11:16:58,670 [INFO] Evaluating model modernmolbert_best_base on dataset ogbg-molmuv with metric roc_auc using head ridge
2026-06-03 11:16:58,673 [INFO] Training model


Logging predictions for ogbg-molmuv dataset
Saving predictions to /Users/skn506/Documents/ModernMolBERT/data/predictions/ogbg-molmuv/modernmolbert_best_base/rf.npy / .npz
[ 1/1] ogbg-molmuv  head=ridge
Shapes: X_test=(9309, 768), y_test=(9309, 17)
Logging predictions for ogbg-molmuv dataset
Saving predictions to /Users/skn506/Documents/ModernMolBERT/data/predictions/ogbg-molmuv/modernmolbert_best_base/ridge.npy / .npz
[ 1/1] ogbg-molmuv  head=knn  skip=disabled (KNN disabled for MUV)


...

Shapes: X_test=(9309, 512), y_test=(9309, 17)


2026-06-03 12:11:47,370 [INFO] Training complete, best CV result: 0.6784242916329769
2026-06-03 12:11:47,387 [INFO] Evaluation complete, test result: 0.6339475534698061
2026-06-03 12:11:47,402 [INFO] Evaluating model modernmolbert_best_standard on dataset ogbg-molmuv with metric roc_auc using head ridge
2026-06-03 12:11:47,405 [INFO] Training model


Logging predictions for ogbg-molmuv dataset
Saving predictions to /Users/skn506/Documents/ModernMolBERT/data/predictions/ogbg-molmuv/modernmolbert_best_standard/rf.npy / .npz
[ 1/1] ogbg-molmuv  head=ridge
Shapes: X_test=(9309, 512), y_test=(9309, 17)
Logging predictions for ogbg-molmuv dataset
Saving predictions to /Users/skn506/Documents/ModernMolBERT/data/predictions/ogbg-molmuv/modernmolbert_best_standard/ridge.npy / .npz


2026-06-03 12:13:20,581 [INFO] Training complete, best CV result: 0.7381123303742653
2026-06-03 12:13:20,598 [INFO] Evaluation complete, test result: 0.739870407629752


[score] wrote checkpoint: outputs/eval/praski_best_standard/checkpoints/ogbg-molmuv__modernmolbert_best_standard.csv
[score] complete: expanded=1, skipped=0, attempted=1, successful_datasets=1, failed_datasets=0
All done
Backfilling checkpoint rows into results.csv for result_not_in_aggregate cases...
  Nothing to backfill.

### rebuild aggregate benchmark frames
$ uv run python scripts/build_benchmark_results_frames.py
Project root: /Users/skn506/Documents/ModernMolBERT
Praski CSV: /Users/skn506/Documents/ModernMolBERT/data/Praski_benchmarking_results/arxiv_preprint_2025_08.csv
Own results root: /Users/skn506/Documents/ModernMolBERT/outputs/eval

Found 4 own results.csv file(s):
  /Users/skn506/Documents/ModernMolBERT/outputs/eval/praski_best_base_standard/results.csv  ->  modernmolbert_best_base
  /Users/skn506/Documents/ModernMolBERT/outputs/eval/praski_best_hetero_span/results.csv  ->  modernmolbert_best_hetero_span
  /Users/skn506/Documents/ModernMolBERT/outputs/eval/praski_best_s

In [8]:
# Re-run this cell after optional scoring and aggregate rebuild.
coverage_after = audit_coverage()
display(coverage_after.groupby(["model", "status"]).size().unstack(fill_value=0))
display(coverage_after.loc[~coverage_after["in_aggregate"]].sort_values(["model", "dataset"]))

status,present
model,
modernmolbert_best_base,25
modernmolbert_best_hetero_span,25
modernmolbert_best_span,25
modernmolbert_best_standard,25


,dataset,model,in_aggregate,in_raw_results,in_checkpoint,has_embedding,status,heads_present,missing_heads,dataset_selector,result_csv,checkpoint_csv,embedding_file
